# 👕 XStore Müşteri Destek Asistanı v2

**Giyim E-Ticaret için Özelleştirilmiş AI Asistan**

🇹🇷 Türkçe + 🇬🇧 English (Detaylı)

---

### 🏪 Mağaza Bilgileri:
- **Mağaza:** XStore
- **Kategori:** Giyim & Moda
- **İade:** 14 gün, ücretsiz
- **Kargo:** 1200₺ üzeri ücretsiz, 2-4 iş günü

In [ ]:
!pip install -q langgraph langchain-core langchain-groq

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "BURAYA_API_KEYINI_YAZ"

In [ ]:
from typing import Dict, TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

def get_llm():
    return ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("✅ Yüklendi!")

In [ ]:
# 🏪 MAĞAZA BİLGİLERİ
STORE = {
    "name": "XStore",
    "category": "Giyim & Moda",
    "return_days": 14,
    "return_free": True,
    "free_shipping_limit": 1200,
    "shipping_days": "2-4 iş günü",
    "shipping_days_en": "2-4 business days",
    "shipping_cost": 49.90,
    "working_hours": "09:00 - 18:00 (Hafta içi)",
    "working_hours_en": "09:00 - 18:00 (Weekdays)",
    "phone": "0850 XXX XX XX",
    "email": "destek@xstore.com",
    "instagram": "@xstore"
}

CATEGORIES = ["product", "order", "return", "payment", "account", "campaign", "general"]

class State(TypedDict):
    query: str
    language: str
    category: str
    sentiment: str
    response: str

In [ ]:
# 🇹🇷 TÜRKÇE PROMPTLAR
TR_PROMPTS = {
    "categorize": f"""Müşteri sorgusunu {STORE['name']} giyim mağazası için kategorize et.

KATEGORİLER:
- product: Ürün, beden, kumaş, renk, stok
- order: Sipariş durumu, kargo, teslimat
- return: İade, değişim, para iadesi
- payment: Ödeme, fatura, taksit
- account: Hesap, şifre, üyelik
- campaign: İndirim, kupon, kampanya
- general: Genel sorular, iletişim

Sadece kategori kodunu yaz: {{query}}""",

    "sentiment": """Duygu analizi yap.

- Pozitif: Teşekkür, memnuniyet
- Nötr: Normal sorular (ÇOĞU MESAJ!)
- Negatif: SADECE küfür, hakaret, "rezalet", "dolandırıcı"

"istiyorum", "bekliyorum", "küçük geldi" = NEGATİF DEĞİL!

Sadece yaz (Pozitif/Nötr/Negatif): {query}""",

    "product": f"""Sen {STORE['name']} müşteri destek asistanısın.

BEDEN REHBERİ:
- XS: 32-34 beden, göğüs 82-86cm
- S: 36-38 beden, göğüs 86-90cm
- M: 38-40 beden, göğüs 90-94cm
- L: 40-42 beden, göğüs 94-98cm
- XL: 42-44 beden, göğüs 98-102cm

İPUÇLARI:
- Rahat istiyorsa bir beden büyük öner
- Ürün sayfasındaki model bilgisine yönlendir

Müşteri: {{query}}""",

    "order": f"""Sen {STORE['name']} müşteri destek asistanısın.

SİPARİŞ & KARGO:
- Siparişler 1-2 iş günü içinde kargoya verilir
- Teslimat: {STORE['shipping_days']}
- {STORE['free_shipping_limit']}₺ üzeri ücretsiz kargo
- Altında {STORE['shipping_cost']}₺ kargo ücreti
- Takip: SMS/email ile bildirilir
- Sipariş durumu: Hesabım > Siparişlerim

Müşteri: {{query}}""",

    "return": f"""Sen {STORE['name']} müşteri destek asistanısın.

İADE POLİTİKASI:
- Süre: {STORE['return_days']} gün
- Ücret: Ücretsiz (kargo bizden)
- Koşul: Etiketli, kullanılmamış, orijinal pakette
- İç giyim/mayo iade edilemez

İADE ADIMLAR:
1. Hesabım > Siparişlerim > İade Talebi
2. İade nedenini seç
3. Kargo kodunu al
4. Kargo şubesine teslim et
5. Para iadesi 3-5 iş günü

BEDEN DEĞİŞİM: Stokta varsa direkt değişim yapılır.

Müşteri: {{query}}""",

    "payment": f"""Sen {STORE['name']} müşteri destek asistanısın.

ÖDEME SEÇENEKLERİ:
- Kredi kartı (tek çekim + taksit)
- Banka kartı
- Havale/EFT
- Kapıda ödeme (+15₺)

TAKSİT:
- 100₺ üzeri: 3 taksit
- 300₺ üzeri: 6 taksit
- 500₺ üzeri: 9 taksit

FATURA: Email ile gönderilir, Hesabım > Siparişlerim'den indirilebilir.

Müşteri: {{query}}""",

    "account": f"""Sen {STORE['name']} müşteri destek asistanısın.

HESAP İŞLEMLERİ:
- Şifre sıfırlama: Giriş > Şifremi Unuttum
- Adres: Hesabım > Adreslerim
- Profil: Hesabım > Profil

ÜYELİK AVANTAJLARI:
- İlk alışverişte %10 indirim
- Özel kampanyalar
- Kolay sipariş takibi

Müşteri: {{query}}""",

    "campaign": f"""Sen {STORE['name']} müşteri destek asistanısın.

AKTİF KAMPANYALAR:
- Yeni üyelere %10 indirim (kod: HOSGELDIN)
- {STORE['free_shipping_limit']}₺ üzeri ücretsiz kargo
- Sezon sonu %50'ye varan indirim

KUPON: Sepette "Kupon Kodu" alanına gir. Her siparişte 1 kupon.

Müşteri: {{query}}""",

    "general": f"""Sen {STORE['name']} müşteri destek asistanısın.

BİLGİLER:
- Çalışma: {STORE['working_hours']}
- Tel: {STORE['phone']}
- Email: {STORE['email']}
- Instagram: {STORE['instagram']}

Müşteri: {{query}}""",

    "escalate": f"""⚠️ Değerli müşterimiz,

Yaşadığınız olumsuz deneyim için özür dileriz. Müşteri hizmetleri ekibimiz sizinle iletişime geçecektir.

Acil: {STORE['phone']}

{STORE['name']} Müşteri Hizmetleri"""
}

# 🇬🇧 ENGLISH PROMPTS (DETAILED)
EN_PROMPTS = {
    "categorize": f"""Categorize this query for {STORE['name']} clothing store.

CATEGORIES:
- product: Size, fabric, color, stock, fit
- order: Order status, shipping, tracking, delivery
- return: Returns, exchanges, refunds
- payment: Payment, invoice, installments
- account: Account, password, membership
- campaign: Discounts, coupons, promotions
- general: Business hours, contact, other

Only write category code: {{query}}""",

    "sentiment": """Analyze sentiment.

- Positive: Thanks, praise, satisfaction
- Neutral: Normal questions (MOST MESSAGES!)
- Negative: ONLY insults, threats, "scam", "terrible", extreme anger

NOT negative: "want to return", "where is my order", "doesn't fit", "waiting"

Only write (Positive/Neutral/Negative): {query}""",

    "product": f"""You are {STORE['name']} customer support assistant.

SIZE GUIDE:
- XS: US 0-2, Chest 32-34" (82-86cm)
- S: US 4-6, Chest 34-35" (86-90cm)
- M: US 8-10, Chest 35-37" (90-94cm)
- L: US 10-12, Chest 37-39" (94-98cm)
- XL: US 14-16, Chest 39-40" (98-102cm)

TIPS:
- For relaxed fit, recommend sizing up
- Refer to model measurements on product page
- Mention fabric stretch when relevant

Customer: {{query}}""",

    "order": f"""You are {STORE['name']} customer support assistant.

ORDER & SHIPPING INFO:
- Orders ship within 1-2 business days
- Delivery: {STORE['shipping_days_en']}
- Free shipping on orders over {STORE['free_shipping_limit']}₺
- Under that: {STORE['shipping_cost']}₺ shipping fee
- Tracking info sent via SMS/email
- Check status: My Account > My Orders
- Cancellation: Only before shipping

Customer: {{query}}""",

    "return": f"""You are {STORE['name']} customer support assistant.

RETURN POLICY:
- Time limit: {STORE['return_days']} days from delivery
- Cost: FREE (we cover return shipping)
- Conditions: Tags attached, unworn, original packaging
- Underwear & swimwear: Non-returnable (hygiene)

RETURN STEPS:
1. Go to My Account > My Orders > Request Return
2. Select return reason
3. Get return shipping code
4. Drop off at nearest shipping point
5. Refund processed in 3-5 business days

SIZE EXCHANGE:
- If in stock: Direct exchange available
- If out of stock: Return + place new order

Customer: {{query}}""",

    "payment": f"""You are {STORE['name']} customer support assistant.

PAYMENT OPTIONS:
- Credit card (one-time or installments)
- Debit card
- Bank transfer (EFT)
- Cash on delivery (+15₺ service fee)

INSTALLMENT PLANS:
- Orders over 100₺: 3 installments
- Orders over 300₺: 6 installments
- Orders over 500₺: 9 installments

INVOICE:
- E-invoice sent via email after purchase
- Download: My Account > My Orders > Download Invoice

Customer: {{query}}""",

    "account": f"""You are {STORE['name']} customer support assistant.

ACCOUNT HELP:
- Password reset: Login > Forgot Password
- Add address: My Account > My Addresses
- Update profile: My Account > Profile
- Delete account: My Account > Settings

MEMBERSHIP BENEFITS:
- 10% off first purchase
- Exclusive campaign notifications
- Easy order tracking
- Save items to wishlist

Customer: {{query}}""",

    "campaign": f"""You are {STORE['name']} customer support assistant.

ACTIVE CAMPAIGNS:
- New members: 10% off (code: WELCOME10)
- Free shipping on orders over {STORE['free_shipping_limit']}₺
- End of season: Up to 50% off selected items

COUPON USAGE:
- Enter code at checkout in "Coupon Code" field
- One coupon per order
- May not apply to already discounted items

Customer: {{query}}""",

    "general": f"""You are {STORE['name']} customer support assistant.

STORE INFORMATION:
- Store: {STORE['name']} (Fashion & Clothing)
- Hours: {STORE['working_hours_en']}
- Phone: {STORE['phone']}
- Email: {STORE['email']}
- Instagram: {STORE['instagram']}

Be helpful and friendly. If unsure, direct to customer service.

Customer: {{query}}""",

    "escalate": f"""⚠️ Dear Customer,

We sincerely apologize for your negative experience. Your satisfaction is our priority, and our customer service team will contact you shortly to resolve this issue.

For immediate assistance: {STORE['phone']}

Thank you for your patience.
{STORE['name']} Customer Service"""
}

LANG = {
    "tr": {
        "name": "Türkçe",
        "cat": {
            "product": "👕 Ürün & Beden",
            "order": "📦 Sipariş & Kargo",
            "return": "🔄 İade & Değişim",
            "payment": "💳 Ödeme & Fatura",
            "account": "🔐 Hesap & Üyelik",
            "campaign": "🎁 Kampanya & İndirim",
            "general": "ℹ️ Genel Bilgi"
        },
        "sent": {
            "positive": "😊 Pozitif",
            "neutral": "😐 Nötr",
            "negative": "😠 Negatif"
        },
        "prompts": TR_PROMPTS
    },
    "en": {
        "name": "English",
        "cat": {
            "product": "👕 Product & Size",
            "order": "📦 Order & Shipping",
            "return": "🔄 Return & Exchange",
            "payment": "💳 Payment & Invoice",
            "account": "🔐 Account",
            "campaign": "🎁 Deals & Discounts",
            "general": "ℹ️ General"
        },
        "sent": {
            "positive": "😊 Positive",
            "neutral": "😐 Neutral",
            "negative": "😠 Negative"
        },
        "prompts": EN_PROMPTS
    }
}

In [ ]:
def norm_cat(c):
    c = c.lower().strip()
    m = {
        "product": "product", "ürün": "product", "beden": "product", "size": "product",
        "order": "order", "sipariş": "order", "kargo": "order", "shipping": "order",
        "return": "return", "iade": "return", "değişim": "return", "exchange": "return",
        "payment": "payment", "ödeme": "payment", "fatura": "payment", "taksit": "payment",
        "account": "account", "hesap": "account", "şifre": "account",
        "campaign": "campaign", "kampanya": "campaign", "indirim": "campaign",
        "general": "general", "genel": "general"
    }
    return m.get(c, "general")

def norm_sent(s):
    s = s.lower().strip()
    m = {
        "pozitif": "positive", "positive": "positive",
        "nötr": "neutral", "neutral": "neutral",
        "negatif": "negative", "negative": "negative"
    }
    return m.get(s, "neutral")

def detect_language(state: State) -> State:
    if state.get("language") in ["tr", "en"]:
        return {"language": state["language"]}
    prompt = ChatPromptTemplate.from_template(
        "Is this Turkish or English? Reply ONLY 'tr' or 'en': {query}"
    )
    res = (prompt | get_llm()).invoke({"query": state["query"]}).content.strip().lower()
    return {"language": res if res in ["tr", "en"] else "tr"}

def categorize(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["categorize"])
    res = (prompt | get_llm()).invoke({"query": state["query"]}).content
    return {"category": norm_cat(res)}

def analyze_sentiment(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["sentiment"])
    res = (prompt | get_llm()).invoke({"query": state["query"]}).content
    return {"sentiment": norm_sent(res)}

def handle_product(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["product"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_order(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["order"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_return(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["return"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_payment(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["payment"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_account(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["account"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_campaign(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["campaign"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def handle_general(state: State) -> State:
    lang = state.get("language", "tr")
    prompt = ChatPromptTemplate.from_template(LANG[lang]["prompts"]["general"])
    return {"response": (prompt | get_llm()).invoke({"query": state["query"]}).content}

def escalate(state: State) -> State:
    lang = state.get("language", "tr")
    return {"response": LANG[lang]["prompts"]["escalate"]}

def route(state: State) -> str:
    if state["sentiment"] == "negative":
        return "escalate"
    return f"handle_{state['category']}"

In [ ]:
# Graph oluştur
wf = StateGraph(State)

wf.add_node("detect_language", detect_language)
wf.add_node("categorize", categorize)
wf.add_node("analyze_sentiment", analyze_sentiment)
wf.add_node("handle_product", handle_product)
wf.add_node("handle_order", handle_order)
wf.add_node("handle_return", handle_return)
wf.add_node("handle_payment", handle_payment)
wf.add_node("handle_account", handle_account)
wf.add_node("handle_campaign", handle_campaign)
wf.add_node("handle_general", handle_general)
wf.add_node("escalate", escalate)

wf.add_edge("detect_language", "categorize")
wf.add_edge("categorize", "analyze_sentiment")
wf.add_conditional_edges("analyze_sentiment", route, {
    "handle_product": "handle_product",
    "handle_order": "handle_order",
    "handle_return": "handle_return",
    "handle_payment": "handle_payment",
    "handle_account": "handle_account",
    "handle_campaign": "handle_campaign",
    "handle_general": "handle_general",
    "escalate": "escalate"
})

for cat in CATEGORIES:
    wf.add_edge(f"handle_{cat}", END)
wf.add_edge("escalate", END)

wf.set_entry_point("detect_language")
app = wf.compile()

print(f"✅ {STORE['name']} Asistanı hazır! (🇹🇷 TR + 🇬🇧 EN)")

In [ ]:
def ask(query: str, language: str = None):
    state = {"query": query}
    if language:
        state["language"] = language
    
    r = app.invoke(state)
    lang = r["language"]
    
    print("="*70)
    print(f"🛒 {STORE['name']} | {LANG[lang]['name']}")
    print("="*70)
    print(f"👤 Customer: {query}")
    print(f"🏷️ Category: {LANG[lang]['cat'][r['category']]}")
    print(f"📊 Sentiment: {LANG[lang]['sent'][r['sentiment']]}")
    print(f"\n🤖 Assistant:\n")
    print(r['response'])
    print("="*70 + "\n")

---
## 🇹🇷 Türkçe Testler
---

In [ ]:
ask("M beden tişört alacağım ama kalıbı nasıl?")

In [ ]:
ask("Siparişim ne zaman gelir?")

In [ ]:
ask("Ürünü iade etmek istiyorum, nasıl yaparım?")

In [ ]:
ask("İndirim kodu var mı?")

---
## 🇬🇧 English Tests
---

In [ ]:
ask("What size should I get? I'm usually a medium in US sizes.")

In [ ]:
ask("When will my order arrive?")

In [ ]:
ask("I want to return this shirt, it doesn't fit. What's the process?")

In [ ]:
ask("Do you have any discount codes for new customers?")

In [ ]:
ask("How much is shipping? Is there free shipping?")

In [ ]:
ask("Can I pay in installments?")

In [ ]:
ask("I forgot my password, how do I reset it?")

---
## 🎮 Kendi Sorunu Dene / Try Your Own
---

In [ ]:
ask("Buraya kendi sorunuzu yazın / Write your question here")